# Data Cleaning: iPhone Resale Market Intelligence (USA 2026)

**Judul Penelitian:** Perancangan Dashboard Market Intelligence untuk Analisis Persaingan Listing Resale iPhone pada Platform E-Commerce Menggunakan Vizro

Notebook ini melakukan data preparation terhadap dataset mentah hasil scraping listing resale iPhone, dengan fokus pada:
- Distribusi harga
- Jumlah listing
- Potensi kompetisi antarproduk/platform


## 1. Import Library & Load Data

In [6]:
import pandas as pd
import numpy as np
import re

# Ganti path sesuai lokasi file di komputer kamu
INPUT_PATH = 'ecommerce_iphone_resale_market_intelligence_usa_2026.csv'
OUTPUT_PATH = 'iphone_resale_cleaned.csv'

# encoding='utf-8-sig' untuk menghapus BOM character di header kolom pertama
df = pd.read_csv("data/ecommerce_iphone_resale_market_intelligence_usa_2026.csv", encoding='utf-8-sig')
n_awal = len(df)
df.head()

,title,model_family,generation_number,is_pro,storage_options_gb,storage_gb_numeric,condition,price,wasPrice,price_discount_pct,available,sold,seller,itemLocation,us_state,lastUpdated
0,Apple iPhone 14 128gb RED color (Factory Unloc...,iPhone 14,14.0,False,128GB,128.0,Used,329.99,NaN,NaN,NaN,NaN,Seller 019,"Granada Hills, California, United States",CA,2026-01-02
1,Lot of 10 Apple iPhone 12 Pro 128GB Unlocked M...,iPhone 12 Pro,12.0,True,128GB,128.0,Used,3160.00,NaN,NaN,NaN,NaN,Seller 332,"Winter Park, Florida, United States",FL,2026-03-01
2,Apple iPhone 14 Pro MAX 128GB FULLY Unlocked,iPhone 14 Pro Max,14.0,True,128GB,128.0,Used,726.99,NaN,NaN,3.0,NaN,Seller 019,"Granada Hills, California, United States",CA,2025-12-12
3,Apple iPhone 14 Pro MAX 256gb Space black (Fac...,iPhone 14 Pro Max,14.0,True,256GB,256.0,Used,669.99,NaN,NaN,NaN,NaN,Seller 019,"Granada Hills, California, United States",CA,2025-12-12
4,Apple iPhone 14 Pro MAX 512gb Deep purple (Fac...,iPhone 14 Pro Max,14.0,True,512GB,512.0,Used,659.99,NaN,NaN,2.0,NaN,Seller 019,"Granada Hills, California, United States",CA,2026-03-10


In [7]:
df.info()
print("\nJumlah missing value per kolom:")
print(df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 2371 entries, 0 to 2370
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   title               2371 non-null   str    
 1   model_family        2371 non-null   str    
 2   generation_number   2371 non-null   float64
 3   is_pro              2371 non-null   bool   
 4   storage_options_gb  2168 non-null   str    
 5   storage_gb_numeric  2168 non-null   float64
 6   condition           2371 non-null   str    
 7   price               2371 non-null   float64
 8   wasPrice            134 non-null    float64
 9   price_discount_pct  134 non-null    float64
 10  available           1222 non-null   float64
 11  sold                912 non-null    float64
 12  seller              2371 non-null   str    
 13  itemLocation        2370 non-null   str    
 14  us_state            2370 non-null   str    
 15  lastUpdated         1956 non-null   str    
dtypes: bool(1), float

## 2. Hapus Duplikat Snapshot

Beberapa listing (title + seller + price sama) ter-scrape berulang pada tanggal berbeda.
Ini adalah listing yang sama, bukan listing berbeda — jika tidak ditangani akan menggandakan
hitungan "jumlah listing" pada dashboard. Snapshot yang disimpan adalah yang `lastUpdated` paling baru.

In [8]:
df['lastUpdated_dt'] = pd.to_datetime(df['lastUpdated'], errors='coerce')
df = df.sort_values('lastUpdated_dt', na_position='first')
df = df.drop_duplicates(subset=['title', 'seller', 'price'], keep='last')
n_after_dedup = len(df)
df = df.reset_index(drop=True)

print(f"Baris awal            : {n_awal}")
print(f"Setelah hapus duplikat: {n_after_dedup} (dihapus {n_awal - n_after_dedup} duplikat snapshot)")

Baris awal            : 2371
Setelah hapus duplikat: 2362 (dihapus 9 duplikat snapshot)


## 3. Perbaikan `model_family` & `generation_number`

In [9]:
# Perbaikan typo penulisan model
df['model_family'] = df['model_family'].str.replace(
    'iPhone 12Pro Max', 'iPhone 12 Pro Max', regex=False
)
df['model_family'] = df['model_family'].str.strip()

# generation_number -> integer (selalu bilangan bulat, misal 12.0 -> 12)
df['generation_number'] = df['generation_number'].astype(int)

df['model_family'].value_counts()

model_family
iPhone 12            406
iPhone 14            335
iPhone 13            330
iPhone 13 Mini       294
iPhone 14 Pro Max    235
iPhone 17 Pro Max    159
iPhone 17            140
iPhone 14 Plus       134
iPhone 12 Pro Max    128
iPhone 16            111
iPhone 15             44
iPhone 12 Pro         10
iPhone 15 Pro          9
iPhone 15 Plus         8
iPhone 12 Mini         7
iPhone 14 Pro          4
iPhone 13 Pro          3
iPhone 13 Pro Max      3
iPhone 17 Pro          1
iPhone 15 Pro Max      1
Name: count, dtype: int64

## 4. Koreksi `storage_gb_numeric`

Ditemukan beberapa anomali pada kolom kapasitas penyimpanan:
- Nilai RAM tertukar dengan storage (contoh: iPhone 17 Pro Max "12GB+256GB" tersimpan sebagai 12GB, seharusnya 256GB)
- Typo angka (125GB → seharusnya 128GB)
- Nilai tidak valid yang tidak bisa dipastikan (28GB, 50GB) → diberi label "Tidak Diketahui" daripada ditebak

In [10]:
VALID_STORAGE = {16, 32, 64, 128, 256, 512, 1024, 2048}

def fix_storage(row):
    val = row['storage_gb_numeric']
    opts = row['storage_options_gb']
    if pd.isna(val):
        return np.nan
    if val in VALID_STORAGE:
        return val
    # kasus RAM tertukar storage, ambil token valid dari storage_options_gb
    if pd.notna(opts):
        tokens = re.findall(r'(\d+)\s*GB', str(opts))
        valid_tokens = [int(t) for t in tokens if int(t) in VALID_STORAGE]
        if valid_tokens:
            return float(min(valid_tokens))
        # koreksi typo umum: 125 -> 128
        corrected = [128 if t == '125' else int(t) for t in tokens]
        valid_corrected = [t for t in corrected if t in VALID_STORAGE]
        if valid_corrected:
            return float(min(valid_corrected))
    return np.nan

df['storage_flag_corrected'] = ~df['storage_gb_numeric'].isin(VALID_STORAGE) & df['storage_gb_numeric'].notna()
df['storage_gb_numeric'] = df.apply(fix_storage, axis=1)

# Kategori storage untuk dashboard
df['storage_category'] = df['storage_gb_numeric'].apply(
    lambda x: f"{int(x)}GB" if pd.notna(x) else "Tidak Diketahui"
)

print(f"Storage dikoreksi: {df['storage_flag_corrected'].sum()} baris")
df['storage_category'].value_counts()

Storage dikoreksi: 5 baris


storage_category
128GB              1079
256GB               444
64GB                320
512GB               235
Tidak Diketahui     204
1024GB               60
2048GB               20
Name: count, dtype: int64

## 5. Standardisasi `condition` & Pengelompokan

In [11]:
df['condition'] = df['condition'].str.strip()

def group_condition(c):
    if c == 'New':
        return 'New'
    if c == 'Open Box':
        return 'Open Box'
    if 'Refurbished' in c:
        return 'Refurbished'
    if c == 'For Parts Or Not Working':
        return 'For Parts'
    return 'Used'

df['condition_group'] = df['condition'].apply(group_condition)
df['condition_group'].value_counts()

condition_group
Used           1104
New             682
Refurbished     241
Open Box        228
For Parts       107
Name: count, dtype: int64

## 6. Flag Listing Bundle/Lot

Listing seperti *"Lot of 10 Apple iPhone 12 Pro"* atau judul yang menyebut banyak model sekaligus
(*"iPhone 14 Pro Max/13/12 Mini..."*) memuat **harga borongan**, bukan harga per unit.
Ditandai dengan flag, bukan dihapus, supaya bisa difilter interaktif di dashboard Vizro.

In [12]:
lot_mask = df['title'].str.contains(r'\blot\b', case=False, na=False)

# deteksi title yang menyebut >1 model iPhone (angka generasi 12-17,
# tidak diikuti huruf G agar tidak salah tangkap "128GB")
pat_multi_model = re.compile(r'(?<![\d])(1[2-7])(?!\s*[Gg])(?=\s|/|,|$)')

def count_models(title):
    return len(set(pat_multi_model.findall(title)))

df['n_models_in_title'] = df['title'].apply(count_models)
df['is_bundle_listing'] = lot_mask | (df['n_models_in_title'] > 1)
df = df.drop(columns=['n_models_in_title'])

print(f"Ditandai bundle/lot: {df['is_bundle_listing'].sum()} baris")
df[df['is_bundle_listing']][['title', 'price']].head()

Ditandai bundle/lot: 68 baris


,title,price
14,Lot of 2 Apple iPhone 13 / Apple iPhone 13 min...,129.95
92,Lot of 10 iPhone 14 Plus 128GB Unlocked (Mint ...,3500.00
95,Lot of 4 iPhone 14 128GB/256GB Unlocked (Great...,1200.00
198,Preowned - Unlocked - Apple iPhone 12 Pro Max ...,349.99
262,Apple iPhone 16 A3081 eSIM 5G 128GB Unlocked C...,515.55


## 7. Flag Harga Mencurigakan

Ditemukan pola listing *"iPhone ... with TikTok/Capcut installed"* dengan harga tidak wajar
(sampai \$5.000 untuk iPhone 13 bekas biasa) — pola dikenal di eBay untuk jualan akun media sosial,
bukan mencerminkan nilai wajar unit HP.

In [13]:
df['is_suspicious_price'] = df['title'].str.contains(r'tiktok|capcut', case=False, na=False)

print(f"Ditandai harga mencurigakan: {df['is_suspicious_price'].sum()} baris")
df[df['is_suspicious_price']][['title', 'price']].sort_values('price', ascending=False).head()

Ditandai harga mencurigakan: 32 baris


,title,price
31,Apple iPhone 13 - 256 GB - TikTok,5000.0
180,iphone 13 unlocked perfect condition with tiktok,5000.0
89,TikTok Apple iPhone 12 Pro (Unlocked),5000.0
86,iPhone 12 Pro With TikTok App,5000.0
358,iphone 13 With TikTok,5000.0


## 8. Flag Kelengkapan Data

Kolom seperti `wasPrice`, `available`, `sold`, `lastUpdated` memang tidak selalu diisi seller.
NaN di sini berarti "tidak diinformasikan", bukan data yang salah/hilang — jadi tidak diimputasi,
hanya diberi flag boolean supaya jelas mana yang punya info lengkap.

In [14]:
df['has_discount'] = df['wasPrice'].notna()
df['available_disclosed'] = df['available'].notna()
df['sold_disclosed'] = df['sold'].notna()
df['date_disclosed'] = df['lastUpdated_dt'].notna()

# itemLocation/us_state kosong (1 baris) -> isi label eksplisit
df['itemLocation'] = df['itemLocation'].fillna('Tidak Diketahui')
df['us_state'] = df['us_state'].fillna('Tidak Diketahui')

## 9. Kolom Turunan untuk Analisis Market Intelligence

In [15]:
# Harga per GB (hanya untuk listing non-bundle dengan storage diketahui)
df['price_per_gb'] = np.where(
    (df['storage_gb_numeric'].notna()) & (~df['is_bundle_listing']),
    df['price'] / df['storage_gb_numeric'],
    np.nan
)

# Bucket harga untuk analisis distribusi
bins = [0, 200, 400, 600, 900, 1300, np.inf]
labels = ['<$200', '$200-399', '$400-599', '$600-899', '$900-1299', '$1300+']
df['price_bucket'] = pd.cut(df['price'], bins=bins, labels=labels, right=False)

df[['price', 'price_bucket', 'price_per_gb']].head()

,price,price_bucket,price_per_gb
0,2500.00,$1300+,4.882812
1,769.99,$600-899,3.007773
2,339.99,$200-399,2.656172
3,395.99,$200-399,3.093672
4,238.99,$200-399,1.867109


## 10. Rapikan Kolom Tanggal & Susun Ulang Kolom

In [16]:
df['lastUpdated'] = df['lastUpdated_dt'].dt.strftime('%Y-%m-%d')
df = df.drop(columns=['lastUpdated_dt'])

df.insert(0, 'listing_id', range(1, len(df) + 1))

col_order = [
    'listing_id', 'title', 'model_family', 'generation_number', 'is_pro',
    'storage_gb_numeric', 'storage_category', 'storage_options_gb', 'storage_flag_corrected',
    'condition', 'condition_group',
    'price', 'price_bucket', 'price_per_gb',
    'wasPrice', 'price_discount_pct', 'has_discount',
    'available', 'available_disclosed', 'sold', 'sold_disclosed',
    'seller', 'itemLocation', 'us_state',
    'lastUpdated', 'date_disclosed',
    'is_bundle_listing', 'is_suspicious_price'
]
df = df[col_order]
df.head()

,listing_id,title,model_family,generation_number,is_pro,storage_gb_numeric,storage_category,storage_options_gb,storage_flag_corrected,condition,...,available_disclosed,sold,sold_disclosed,seller,itemLocation,us_state,lastUpdated,date_disclosed,is_bundle_listing,is_suspicious_price
0,1,Apple iPhone 12 Pro Max - 512 GB - Blue (Unloc...,iPhone 12 Pro Max,12,True,512.0,512GB,512GB,False,Used,...,False,NaN,False,Seller 697,"Ashtabula, Ohio, United States",OH,NaN,False,False,False
1,2,open box Apple iPhone 14 Pro MAX 256gb purple ...,iPhone 14 Pro Max,14,True,256.0,256GB,256GB,False,Open Box,...,False,NaN,False,Seller 019,"Granada Hills, California, United States",CA,NaN,False,False,False
2,3,open box Apple iPhone 14 128gb purple color Fa...,iPhone 14,14,False,128.0,128GB,128GB,False,Open Box,...,False,NaN,False,Seller 019,"Granada Hills, California, United States",CA,NaN,False,False,False
3,4,open box Apple iPhone 14+ plus 128gb Black col...,iPhone 14,14,False,128.0,128GB,128GB,False,Open Box,...,False,NaN,False,Seller 019,"Granada Hills, California, United States",CA,NaN,False,False,False
4,5,Apple iPhone 12 128GB Unlocked MODEL A2172 Blu...,iPhone 12,12,False,128.0,128GB,128GB,False,Open Box,...,True,NaN,False,Seller 019,"Granada Hills, California, United States",CA,NaN,False,False,False


## 11. Simpan Output & Ringkasan

In [17]:
df.to_csv(OUTPUT_PATH, index=False)

print(f"Baris awal              : {n_awal}")
print(f"Setelah hapus duplikat  : {n_after_dedup}  (dihapus {n_awal - n_after_dedup} duplikat snapshot)")
print(f"Baris akhir tersimpan   : {len(df)}")
print(f"Ditandai bundle/lot     : {df['is_bundle_listing'].sum()}")
print(f"Ditandai harga mencurigakan (TikTok/Capcut): {df['is_suspicious_price'].sum()}")
print(f"Storage dikoreksi       : {df['storage_flag_corrected'].sum()}")
print(f"Storage tetap 'Tidak Diketahui': {(df['storage_category']=='Tidak Diketahui').sum()}")
print(f"\nJumlah listing 'bersih' utk analisis harga per produk (bukan bundle & bukan mencurigakan): "
      f"{(~df['is_bundle_listing'] & ~df['is_suspicious_price']).sum()}")

Baris awal              : 2371
Setelah hapus duplikat  : 2362  (dihapus 9 duplikat snapshot)
Baris akhir tersimpan   : 2362
Ditandai bundle/lot     : 68
Ditandai harga mencurigakan (TikTok/Capcut): 32
Storage dikoreksi       : 5
Storage tetap 'Tidak Diketahui': 204

Jumlah listing 'bersih' utk analisis harga per produk (bukan bundle & bukan mencurigakan): 2262


In [21]:
iphone12pro = df[df["model_family"] == "iPhone 12 Pro"]

iphone12pro[["price"]].sort_values("price")

,price
1419,314.11
1514,314.11
1194,332.01
1329,332.08
1434,332.08
1312,332.08
208,2500.00
699,3160.00
89,5000.00
86,5000.00
